In [1]:
# Remove doublings from table and estimate them
# FM 20/10/2025

In [2]:
import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo

import pandas as pd
import numpy as np_orch
from collections import defaultdict

/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-10-20 13:07:03.517276: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
fileorch='midis/liszt_classical_archives-0/orchestra.mid'

In [4]:
print(f"Orchestra file: {fileorch}")
dforch = midi_to_dataframe(fileorch)
dforch = dforch.sort_values(
    ['onset in quarter notes', 'duration in quarter notes', 'track number'],
    ascending=[True, True, True]
)
print(dforch.columns)
print("Table:\n", dforch)
np_orch = dforch.to_numpy()
mapping = learn_quaterna_mapping(np_orch, ytarget="track-channel")
print("Mapping:", mapping)

Orchestra file: midis/liszt_classical_archives-0/orchestra.mid
Index(['track number', 'track name', 'channel', 'program',
       'onset in quarter notes', 'duration in quarter notes', 'pitch',
       'velocity'],
      dtype='object')
Table:
       track number   track name  channel  program  onset in quarter notes  \
0                1        Flute        1       73                     0.0   
1                1        Flute        1       73                     0.0   
585              2         Oboe        2       68                     0.0   
586              2         Oboe        2       68                     0.0   
1206             3     Clarinet        3       71                     0.0   
...            ...          ...      ...      ...                     ...   
1205             2         Oboe        2       68                  1574.0   
4791            10       Violin       10       40                  1574.0   
5491            11        Viola       11       41               

In [5]:
def reduce_df_with_transform(df, tol=1.0, transformations=None):
    df_reduced = df
    df_hashed = defaultdict(list)
    count_match = {
        'transformed': 0,
        'direct': 0
    }

    for func, kwargs in transformations:
        df_reduced[f"{func.__name__}_{kwargs}"] = False

    todrop = []

    for idx, note in df_reduced.iterrows():
        key = (note['onset in quarter notes'], note['pitch'])        

        matched = False
        if transformations:
            for func, kwargs in transformations:
                if not matched:
                    transformed = func(note, **kwargs)
                    for t in transformed:
                        key_t = (t['onset in quarter notes'], t['pitch'])
                        if len(df_hashed[key_t]) > 0: # if something matches with the transformation
                            for candidate_match_index in df_hashed[key_t]:
                                candidate_match = df_reduced.loc[candidate_match_index]
                                dur_diff = abs(candidate_match['duration in quarter notes'] - note['duration in quarter notes'])
                                if dur_diff <= note['duration in quarter notes'] * tol:
                                    df_reduced.at[candidate_match_index, f"{func.__name__}_{kwargs}"] = True
                                    matched = True
                                    count_match['transformed'] += 1
                                    todrop.append(idx)
                                    break
                            if not matched:
                                df_hashed[key_t].append(idx)
                                matched = True
                                count_match['direct'] += 1
                                break

        if not matched:
            df_hashed[key].append(idx)
            count_match['direct'] += 1

    print(count_match)
    print(f"Original size: {df_reduced.shape}")
    print(f"Dropping {todrop}")
    df_reduced.drop(todrop, inplace=True)
    print(f"Size after drop: {df_reduced.shape}")

    return df_reduced


In [6]:
def transpose(note, n_semitones=12):
    # Example: transpose pitch
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [7]:
def split_duration(note, smallest_unit=1.0):
    # Split a note into a set of notes of duration smallest_unit
    # ['track number', 'track name', 'channel', 'program', 'onset in quarter notes', 'duration in quarter notes', 'pitch', 'velocity']
    transformed_notes = []
    dur = note['duration in quarter notes']
    n = int(dur // smallest_unit)
    for i in range(n):
        new_note = note.copy()
        new_note['onset in quarter notes'] = note['onset in quarter notes'] + smallest_unit*i
        new_note['duration in quarter notes'] = smallest_unit
        transformed_notes.append(new_note)
    if dur - smallest_unit*n > 0:
        new_note = note.copy()
        new_note['onset in quarter notes'] = note['onset in quarter notes'] + smallest_unit*n
        new_note['duration in quarter notes'] = dur - smallest_unit*n
        transformed_notes.append(new_note)
    return transformed_notes

In [8]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]


In [9]:
df_reduced = reduce_df_with_transform(dforch, tol=0.2, transformations=transformations)

{'transformed': 2511, 'direct': 4106}
Original size: (6617, 14)
Dropping [585, 586, 1207, 1676, 2776, 2777, 3775, 3776, 4792, 5492, 2370, 587, 588, 1208, 1209, 1678, 2778, 2779, 2780, 3777, 3778, 4793, 5493, 589, 1211, 1680, 2371, 2781, 2782, 2783, 3779, 3780, 3781, 4794, 5494, 591, 1212, 1213, 1682, 2374, 2784, 2785, 2786, 3782, 3783, 3784, 4795, 5495, 593, 594, 1215, 1684, 2375, 2787, 2788, 2789, 3785, 3786, 6085, 595, 596, 1217, 1686, 2376, 2790, 2791, 2792, 3787, 3788, 4797, 6086, 597, 598, 1219, 1688, 2377, 2793, 2794, 2795, 3789, 3790, 4798, 6087, 599, 600, 1220, 1221, 1690, 2378, 2628, 2629, 2796, 2798, 3791, 3793, 4799, 6088, 3794, 3795, 3796, 3797, 3798, 2381, 1222, 3799, 3800, 1693, 3801, 1696, 3802, 3803, 3804, 1698, 3805, 3806, 3807, 3808, 5507, 3809, 1703, 2382, 2383, 3810, 1223, 1224, 1705, 2384, 2385, 3811, 3812, 1225, 1707, 2386, 2387, 3813, 3814, 1226, 3815, 3816, 1227, 5513, 3817, 3818, 3819, 3820, 1228, 1229, 1714, 3821, 4812, 1230, 1231, 1715, 3822, 4813, 1232, 1233

In [10]:
dforch

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
0,1,Flute,1,73,0.0,2.0,82,60,True,False,False,False,False,False
1,1,Flute,1,73,0.0,2.0,88,60,True,True,False,False,False,False
1206,3,Clarinet,3,71,0.0,2.0,67,60,True,False,False,False,False,False
1675,4,Bassoon,4,70,0.0,2.0,48,60,False,False,False,True,False,False
2775,9,Violin,10,40,0.0,2.0,67,60,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4789,10,Violin,10,40,1570.0,4.0,60,60,False,False,False,False,False,False
4790,10,Violin,10,40,1570.0,4.0,68,60,False,False,False,False,False,False
1204,2,Oboe,2,68,1574.0,1.0,75,60,True,False,False,False,False,False
1205,2,Oboe,2,68,1574.0,1.0,79,60,True,False,False,False,False,False


In [11]:
df_reduced

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
0,1,Flute,1,73,0.0,2.0,82,60,True,False,False,False,False,False
1,1,Flute,1,73,0.0,2.0,88,60,True,True,False,False,False,False
1206,3,Clarinet,3,71,0.0,2.0,67,60,True,False,False,False,False,False
1675,4,Bassoon,4,70,0.0,2.0,48,60,False,False,False,True,False,False
2775,9,Violin,10,40,0.0,2.0,67,60,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4789,10,Violin,10,40,1570.0,4.0,60,60,False,False,False,False,False,False
4790,10,Violin,10,40,1570.0,4.0,68,60,False,False,False,False,False,False
1204,2,Oboe,2,68,1574.0,1.0,75,60,True,False,False,False,False,False
1205,2,Oboe,2,68,1574.0,1.0,79,60,True,False,False,False,False,False


In [12]:
df_reduced[df_reduced["transpose_{'n_semitones': 46}"]==True]

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
329,1,Flute,1,73,419.0,1.0,84,60,True,True,True,False,False,False


In [13]:
save_midi_with_exact_timing_structure(df_reduced, 'midis/example_reduced.mid', reference_midi_path=fileorch)


=== SAVING WITH EXACT TIMING STRUCTURE ===
Extracting timing structure from midis/liszt_classical_archives-0/orchestra.mid
ticks_per_beat: 1024
Found 0 timing events:
Using ticks_per_beat: 1024
Applying key signature fix for MuseScore compatibility...
✓ Added consistent key signatures to 10 instrument tracks
✅ Saved midis/example_reduced.mid with exact timing structure preserved
   - 0 timing events preserved
   - 10 instrument tracks created
   - Key signatures fixed for MuseScore compatibility


In [14]:
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import numpy as np
import time
import joblib

KERAS_AVAILABLE = False

def estimate_transform(df, ytarget="transpose_{'n_semitones': 12}", model="XGBoost", pipeline_path=""):
    """
    FM
    """
    
    # Define available classifiers and their names
    classifiers_map = {
        "XGBoost": XGBClassifier(),
        "RandomForest": RandomForestClassifier(),
        "DecisionTree": DecisionTreeClassifier(),
        "NearestNeighbors": KNeighborsClassifier(1),
        "MLP3": MLPClassifier(hidden_layer_sizes=(128, 128, 128)),
        "NaiveBayes": GaussianNB(),
        "MLP1": MLPClassifier(),
        "AdaBoost": AdaBoostClassifier(),
    }
    
    if KERAS_AVAILABLE:
        classifiers_map["LSTMClassifier"] = KerasClassifierWrapper(build_lstm_classifier)
        classifiers_map["TransformerClassifier"] = KerasClassifierWrapper(build_transformer_classifier)

    # Check if the requested model is available
    if model not in classifiers_map:
        if "LSTMClassifier" in model or "TransformerClassifier" in model:
            print(f"Error: Keras is not available. Cannot use {model}.")
            return
        else:
            print(f"Error: Invalid model name '{model}'. Available models are: {list(classifiers_map.keys())}")
            return
            
    clf_name = model
    clf = classifiers_map[clf_name]

    # Load and process source file
    print(f"Learning {ytarget}")
    df = df.sort_values(
        ['onset in quarter notes', 'duration in quarter notes', 'track number'],
        ascending=[True, True, True]
    )
    nmat = df.to_numpy()
    mapping = learn_quaterna_mapping(nmat, ytarget)
    print("Mapping:", mapping)
    
    X, _ = defineXy(nmat, ytarget)
    y = df[ytarget].to_numpy()
    print("Labels", np.unique(y))
    print("Number of events:", X.shape[0])
    print("Last onset at", X[X.shape[0] - 1, 0])

    if len(np.unique(y)) == 1:
        print("Only one label in target variable")
        return
    
    # Partition the dataset
    X_train, X_test, y_train, y_test, le = split_and_encode(X, y, test_size=0.2, random_state=42)
    X_train_f, _, y_train_f, _, le_f = split_and_encode(X, y, test_size=0, random_state=42)
    
    # Train and predict with the specified classifier
    print(f"\n--------- {clf_name} ---------")
    start = time.time()
    
    if clf_name in ["LSTMClassifier", "TransformerClassifier"]:
        clf_pipeline = make_pipeline(StandardScaler(), clf)
    else:
        clf_pipeline = clf
        
    clf_pipeline.fit(X_train, y_train)
    score = clf_pipeline.score(X_test, y_test) # This is accuracy (?)
    # TODO: Here and everywhere, use more scoring functions (prec, rec, f1, acc)
    end = time.time()
    
    print("Train Time (sec):", f"{end - start:.4f}")
    print("Score on Test (20%):", f"{score:.4f}")
    #
    # Save all needed artifacts
    if pipeline_path!="": 
        artifact = {
            "pipeline": clf_pipeline,                 # pipeline
            "label_encoder": le_f,           # label encoder for final training
            "mapping": mapping,              # quaterna reconstruction mapping
            "ytarget": ytarget,              # 
            }

        joblib.dump(artifact, pipeline_path)
        print(f"[AMO-XGB SAVE] Pipeline saved: {pipeline_path}")
    #

In [ ]:
for func, kwargs in transformations:
    ytarget = f"{func.__name__}_{kwargs}"
    estimate_transform(df_reduced, ytarget=ytarget, model="XGBoost", pipeline_path="")
    print("\n")

# TODO: Use one model for multi-variate target prediction

Learning transpose_{'n_semitones': 12}
Mapping: [[1 'Flute' 1 73]
 [2 'Oboe' 2 68]
 [3 'Clarinet' 3 71]
 [4 'Bassoon' 4 70]
 [5 'Horn' 5 60]
 [6 'Trumpet' 6 56]
 [9 'Violin' 10 40]
 [10 'Violin' 10 40]
 [11 'Viola' 11 41]
 [12 'Violoncello' 12 42]
 [13 'Contrabass' 13 43]]
Labels [False  True]
Number of events: 4106
Last onset at 1574.0
y_train, test size: 0.2 , labels: [False  True]
y_train, test size: 0 , labels: [False  True]

--------- XGBoost ---------
Train Time (sec): 1.5259
Score on Test (20%): 0.8236


Learning transpose_{'n_semitones': 24}
Mapping: [[1 'Flute' 1 73]
 [2 'Oboe' 2 68]
 [3 'Clarinet' 3 71]
 [4 'Bassoon' 4 70]
 [5 'Horn' 5 60]
 [6 'Trumpet' 6 56]
 [9 'Violin' 10 40]
 [10 'Violin' 10 40]
 [11 'Viola' 11 41]
 [12 'Violoncello' 12 42]
 [13 'Contrabass' 13 43]]
Labels [False  True]
Number of events: 4106
Last onset at 1574.0
y_train, test size: 0.2 , labels: [False  True]
y_train, test size: 0 , labels: [False  True]

--------- XGBoost ---------
Train Time (sec): 0.4